# E5: directional

YOLO26s with one residual depthwise horizontal/vertical convolution block at P3.

Normal full v5, five classes, 80 epochs at 640 by default. E0 is intentionally omitted. No accuracy improvement is assumed.

**Colab:** enable GPU under Runtime. **Kaggle:** enable GPU and Internet in notebook settings. Import this notebook and run cells in order; it includes all custom code. Use a read-access HF_TOKEN secret if your dataset is private. One experiment per GPU session.

Keep the same dataset commit, seed, batch and training settings across comparisons. Full training is not started until the training cell. Estimated time depends on allocated GPU.

## 1. Extract embedded source files

In [ ]:
from pathlib import Path
import sys, os
BASE = Path('/kaggle/working') if Path('/kaggle/working').exists() else (Path('/content') if Path('/content').exists() else Path.cwd())
PACKAGE = BASE / 'roadeeye_experiments'
PACKAGE.mkdir(parents=True, exist_ok=True)
FILES = {'modules.py': '"""Importable checkpoint classes; all preserve input channel count and resolution."""\nimport torch\nfrom torch import nn\nfrom ultralytics.nn.modules import Conv\n\n\nclass WeightedFusion(nn.Module):\n    """Project concatenated branches, learn a normalized weighted sum, add a residual.\n\n    This adapts weighted fusion inside the YOLO PAN neck; it is NOT a full BiFPN.\n    Output channels equal the concatenated input so no parser patch is necessary.\n    """\n    def __init__(self, channels):\n        super().__init__()\n        self.channels = list(channels)\n        width = min(channels)\n        self.project = nn.ModuleList(Conv(c, width, 1) for c in channels)\n        self.weights = nn.Parameter(torch.zeros(len(channels)))\n        self.expand = Conv(width, sum(channels), 1)\n        self.gain = nn.Parameter(torch.tensor(0.1))\n\n    def forward(self, x):\n        parts = x.split(self.channels, dim=1)\n        weights = self.weights.softmax(dim=0)\n        fused = sum(w * layer(part) for w, layer, part in zip(weights, self.project, parts))\n        return x + self.gain * self.expand(fused)\n\n\nclass DirectionalConv(nn.Module):\n    """Residual depthwise horizontal/vertical 7-pixel kernels; not deformable convolution."""\n    def __init__(self, channels, kernel=7):\n        super().__init__()\n        self.horizontal = nn.Conv2d(channels, channels, (1, kernel), padding=(0, kernel // 2),\n                                    groups=channels, bias=False)\n        self.vertical = nn.Conv2d(channels, channels, (kernel, 1), padding=(kernel // 2, 0),\n                                  groups=channels, bias=False)\n        self.fuse = Conv(2 * channels, channels, 1)\n        self.gain = nn.Parameter(torch.tensor(0.1))\n\n    def forward(self, x):\n        return x + self.gain * self.fuse(torch.cat((self.horizontal(x), self.vertical(x)), 1))\n\n\nclass SimAM(nn.Module):\n    def __init__(self, channels=None, epsilon=1e-4):\n        super().__init__()\n        self.epsilon = epsilon\n\n    def forward(self, x):\n        n = max(x.shape[-1] * x.shape[-2] - 1, 1)\n        delta = (x - x.mean(dim=(2, 3), keepdim=True)).square()\n        energy = delta / (4 * (delta.sum(dim=(2, 3), keepdim=True) / n + self.epsilon)) + 0.5\n        return x * energy.sigmoid()\n\n\ndef register_modules():\n    import ultralytics.nn.tasks as tasks\n    for module in (WeightedFusion, DirectionalConv, SimAM):\n        setattr(tasks, module.__name__, module)\n', 'architectures.py': '"""Build E1-E6 and explicitly map original layers for pretrained weight transfer."""\nimport copy\nimport math\nfrom pathlib import Path\n\nimport yaml\n\nROOT = Path(__file__).resolve().parent\nEXPERIMENTS = {\n    \'E1\': \'p2\', \'E2\': \'weighted_fusion\', \'E3\': \'p2_weighted_fusion\',\n    \'E4\': \'yolo26m\', \'E5\': \'directional\', \'E6\': \'single_simam\',\n}\n\n\ndef build_config(experiment, simam_base=\'E3\'):\n    if experiment not in EXPERIMENTS or simam_base not in {\'E1\', \'E2\', \'E3\', \'E4\', \'E5\'}:\n        raise ValueError(\'Expected E1-E6; E6 base must be E1-E5\')\n    base = simam_base if experiment == \'E6\' else experiment\n    p2, weighted = base in {\'E1\', \'E3\'}, base in {\'E2\', \'E3\'}\n    cfg = yaml.safe_load((ROOT / \'yolo26_base.yaml\').read_text(encoding=\'utf-8\'))\n    cfg[\'scale\'] = scale = \'m\' if base == \'E4\' else \'s\'\n    _, width, maximum = cfg[\'scales\'][scale]\n    scaled = lambda c: math.ceil(min(c, maximum) * width / 8) * 8\n    original = cfg[\'backbone\'] + cfg[\'head\']\n    nodes, channels, mapped, transfer = [], [], {}, {}\n\n    def append(f, repeat, module, args, c):\n        idx = len(nodes)\n        nodes.append([f, repeat, module, args])\n        channels.append(c)\n        return idx\n\n    def fusion(f, branch_channels):\n        return append(f, 1, \'WeightedFusion\', [branch_channels], sum(branch_channels))\n\n    # Keep original backbone, then remap every absolute head reference as layers are inserted.\n    for old_idx, (f, n, module, args) in enumerate(original[:-1]):\n        args = copy.deepcopy(args)\n        refs = [mapped[x] if x != -1 else -1 for x in f] if isinstance(f, list) else mapped.get(f, f)\n        if module == \'Concat\':\n            incoming = [channels[i] for i in refs]\n            c = sum(incoming)\n        elif module == \'nn.Upsample\':\n            c = channels[refs]\n        else:\n            c = scaled(args[0])\n        idx = append(refs, n, module, args, c)\n        transfer[idx] = old_idx\n        if module == \'Concat\' and weighted:\n            idx = fusion(-1, incoming)\n        if old_idx == 16 and base == \'E5\':\n            idx = append(-1, 1, \'DirectionalConv\', [c, 7], c)\n        if old_idx == 16 and experiment == \'E6\' and not p2:\n            idx = append(-1, 1, \'SimAM\', [c], c)\n        mapped[old_idx] = idx\n\n    detection = [mapped[16], mapped[19], mapped[22]]\n    if p2:\n        up = append(mapped[16], 1, \'nn.Upsample\', [None, 2, \'nearest\'], channels[mapped[16]])\n        incoming = [channels[up], channels[mapped[2]]]\n        append([up, mapped[2]], 1, \'Concat\', [1], sum(incoming))\n        if weighted:\n            fusion(-1, incoming)\n        finest = append(-1, 2, \'C3k2\', [128, True], scaled(128))\n        if experiment == \'E6\':\n            finest = append(-1, 1, \'SimAM\', [scaled(128)], scaled(128))\n        # Append P2 LAST: keeps P3-P5 head widths and indices compatible with pretrained YOLO.\n        detection.append(finest)\n    head = append(detection, 1, \'Detect\', [\'nc\'], None)\n    transfer[head] = 23\n    cfg[\'backbone\'], cfg[\'head\'] = nodes[:11], nodes[11:]\n    cfg[\'source_layer_map\'] = transfer\n    cfg[\'experiment\'] = experiment\n    cfg[\'simam_base\'] = simam_base if experiment == \'E6\' else None\n    return cfg\n\n\ndef write_config(experiment, directory, simam_base=\'E3\', pretrained=\'auto\'):\n    cfg = build_config(experiment, simam_base)\n    cfg[\'pretrained_source\'] = f"yolo26{cfg[\'scale\']}.pt" if pretrained == \'auto\' else pretrained\n    directory = Path(directory)\n    directory.mkdir(parents=True, exist_ok=True)\n    suffix = f\'_{simam_base}\' if experiment == \'E6\' else \'\'\n    target = directory / f"yolo26{cfg[\'scale\']}_{experiment}{suffix}.yaml"\n    target.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding=\'utf-8\')\n    return target\n', 'dataset.py': '"""Resolve the normal v5 Hub snapshot; repair machine-specific YAML paths locally."""\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\n\nimport yaml\n\nREPO_ID = \'AkumaDachi/roadeeye-sewer-defects-v5\'\nREVISION = \'d88c0ed4f0fd39253eb3866bf2feb9dcf808bae5\'\nNAMES = [\'crack\', \'corrosion_rust\', \'sediment_deposit\', \'root_intrusion\', \'joint_defect\']\nCOUNTS = {\'train\': 9486, \'val\': 2014, \'test\': 1986}\nEXTENSIONS = {\'.jpg\', \'.jpeg\', \'.png\', \'.bmp\', \'.webp\', \'.tif\', \'.tiff\'}\n\n\ndef prepare_dataset(workspace, repo_id=REPO_ID, revision=REVISION, local_data=None,\n                    strict_counts=True):\n    workspace = Path(workspace).resolve()\n    workspace.mkdir(parents=True, exist_ok=True)\n    commit = None\n    if local_data:\n        source = Path(local_data).resolve()\n        source = source.parent if source.is_file() else source\n    else:\n        from huggingface_hub import HfApi, snapshot_download\n        token = os.getenv(\'HF_TOKEN\') or None  # also allows existing local HF authentication\n        info = HfApi().repo_info(repo_id=repo_id, repo_type=\'dataset\', revision=revision, token=token)\n        commit = info.sha\n        snapshot = Path(snapshot_download(\n            repo_id=repo_id, repo_type=\'dataset\', revision=commit, token=token,\n            cache_dir=str(workspace / \'hf_cache\'),\n            allow_patterns=[\'data.yaml\', \'train/*\', \'valid/*\', \'val/*\', \'test/*\',\n                            \'final_dataset_v5/*\', \'README.md\'],\n            ignore_patterns=[\'*.cache\', \'*.pt\'], max_workers=4,\n        ))\n        candidates = [p.parent for p in snapshot.rglob(\'data.yaml\')\n                      if (p.parent / \'train/images\').is_dir() and (p.parent / \'test/images\').is_dir()]\n        if len(candidates) != 1:\n            raise ValueError(f\'Expected one v5 root with data.yaml/train/test; found {candidates}\')\n        source = candidates[0]  # do not resolve individual symlinks out of snapshot structure\n    raw = yaml.safe_load((source / \'data.yaml\').read_text(encoding=\'utf-8\'))\n    names = raw.get(\'names\', [])\n    names = [names.get(i, names.get(str(i))) for i in range(len(names))] if isinstance(names, dict) else names\n    if names != NAMES:\n        raise ValueError(f\'Expected normal v5 five-class taxonomy, got {names}\')\n    cfg = {\'path\': str(source), \'nc\': 5, \'names\': dict(enumerate(NAMES))}\n    counts, hashes, label_warnings = {}, {}, []\n    for split, candidates in {\'train\': [\'train\'], \'val\': [\'valid\', \'val\'], \'test\': [\'test\']}.items():\n        folder = next((source / name for name in candidates if (source / name / \'images\').is_dir()), None)\n        if folder is None or not (folder / \'labels\').is_dir():\n            raise ValueError(f\'Missing image/label directories for {split} in {source}\')\n        images = sorted(p for p in (folder / \'images\').rglob(\'*\') if p.suffix.lower() in EXTENSIONS)\n        if not images:\n            raise ValueError(f\'Empty split: {split}\')\n        counts[split] = len(images)\n        digest = hashlib.sha256()\n        for image in images:\n            relative = image.relative_to(folder / \'images\')\n            label = folder / \'labels\' / relative.with_suffix(\'.txt\')\n            if not label.is_file():\n                raise ValueError(f\'Missing label file: {label}; use an empty file for true background images\')\n            content = label.read_bytes()\n            for line_number, line in enumerate(content.decode(\'utf-8-sig\').splitlines(), 1):\n                if not line.strip():\n                    continue\n                fields = line.split()\n                if len(fields) != 5 or fields[0] not in {\'0\', \'1\', \'2\', \'3\', \'4\'}:\n                    raise ValueError(f\'Invalid YOLO detection annotation: {label}\')\n                coords = list(map(float, fields[1:]))\n                # Match Ultralytics\' 1% coordinate tolerance; do not silently clean the published v5.\n                if not all(math.isfinite(c) and -0.01 <= c <= 1.01 for c in coords):\n                    raise ValueError(f\'Invalid normalized box: {label}\')\n                if min(coords[2:]) <= 0 or not all(0 <= c <= 1 for c in coords):\n                    label_warnings.append({\'split\': split, \'label\': str(label.relative_to(source)),\n                                           \'line\': line_number, \'issue\': \'zero/nonpositive size or tolerance-bound coordinate\'})\n            digest.update(relative.as_posix().encode())\n            digest.update(b\'\\0\' + content + b\'\\0\')\n        hashes[split] = digest.hexdigest()\n        cfg[split] = str(folder / \'images\')\n    if strict_counts and counts != COUNTS:\n        raise ValueError(f\'Unexpected full-v5 counts {counts}; expected {COUNTS}. \'\n                         \'Check the upload. Use --allow-count-mismatch only for deliberate smoke-test fixtures.\')\n    resolved = workspace / \'data_v5_resolved.yaml\'\n    resolved.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding=\'utf-8\')\n    provenance = {\'repo_id\': None if local_data else repo_id, \'requested_revision\': revision,\n                  \'resolved_commit\': commit, \'local_root\': str(source), \'counts\': counts,\n                  \'filename_label_sha256\': hashes, \'preprocessing\': \'normal v5, no CLAHE\',\n                  \'label_warnings\': label_warnings}\n    (workspace / \'dataset_provenance.json\').write_text(json.dumps(provenance, indent=2), encoding=\'utf-8\')\n    print(f\'Dataset ready: {counts}; revision: {commit or "local"}\', flush=True)\n    if label_warnings:\n        print(f\'WARNING: {len(label_warnings)} suspicious label rows preserved for consistent comparisons; \'\n              \'see dataset_provenance.json. Dataset was not relabelled.\', flush=True)\n    return resolved, provenance\n', 'experiment_trainer.py': '"""Preserve pretrained transfer across inserted layers, including inside Trainer rebuilds."""\nimport json\nfrom pathlib import Path\n\nfrom ultralytics import YOLO\nfrom ultralytics.models.yolo.detect import DetectionTrainer\n\nfrom modules import register_modules\n\n\ndef transfer_pretrained(target, source_path):\n    source = YOLO(str(source_path)).model.cpu().float()\n    mapping = target.yaml[\'source_layer_map\']\n    target_state, source_state = target.state_dict(), source.state_dict()\n    transferred, skipped = {}, []\n    for key, value in target_state.items():\n        pieces = key.split(\'.\')\n        new_index = int(pieces[1])\n        old_index = mapping.get(new_index, mapping.get(str(new_index)))\n        source_key = \'.\'.join([\'model\', str(old_index), *pieces[2:]])\n        if old_index is not None and source_key in source_state and source_state[source_key].shape == value.shape:\n            transferred[key] = source_state[source_key]\n        else:\n            skipped.append(key)\n    target.load_state_dict(transferred, strict=False)\n    # Every unchanged backbone tensor must transfer; fail rather than silently start it from scratch.\n    missing_backbone = [k for k in skipped if int(k.split(\'.\')[1]) < 11]\n    if missing_backbone:\n        raise ValueError(f\'Incompatible pretrained backbone: {missing_backbone[:5]}\')\n    report = {\'source\': str(source_path), \'loaded_tensors\': len(transferred),\n              \'total_tensors\': len(target_state), \'fresh_tensors\': skipped,\n              \'loaded_state_elements\': sum(v.numel() for v in transferred.values()),\n              \'all_backbone_tensors_loaded\': True}\n    print(f"Mapped pretrained transfer: {len(transferred)}/{len(target_state)} tensors; full backbone loaded")\n    return report\n\n\nclass ExperimentTrainer(DetectionTrainer):\n    def check_resume(self, overrides):\n        super().check_resume(overrides)\n        if self.resume and overrides.get(\'data\'):\n            # A restarted cloud session may have the same YAML filename but a new cache root.\n            self.args.data = overrides[\'data\']\n\n    def get_model(self, cfg=None, weights=None, verbose=True):\n        register_modules()\n        model = super().get_model(cfg=cfg, weights=weights, verbose=verbose)\n        source = model.yaml.get(\'pretrained_source\')\n        if weights is None and source:\n            report = transfer_pretrained(model, source)\n            (Path(self.save_dir) / \'pretrained_transfer.json\').write_text(json.dumps(report, indent=2), encoding=\'utf-8\')\n        return model\n', 'run_experiment.py': '"""Common E1-E6 runner. Training uses validation; test evaluation is explicit."""\nimport argparse\nimport json\nimport platform\nimport shutil\nimport time\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom uuid import uuid4\n\nimport torch\nimport ultralytics\nfrom ultralytics import YOLO\nfrom ultralytics.utils.torch_utils import init_seeds\n\nfrom architectures import EXPERIMENTS, write_config\nfrom dataset import prepare_dataset, REPO_ID, REVISION\nfrom experiment_trainer import ExperimentTrainer\nfrom modules import register_modules\n\nROOT = Path(__file__).resolve().parent\nDEFAULT_LOCAL_DATA = ROOT.parent / \'final_dataset_v5\'\n\n\ndef arguments(argv=None, default_experiment=None):\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--experiment\', choices=EXPERIMENTS, default=default_experiment, required=default_experiment is None)\n    p.add_argument(\'--simam-base\', choices=[\'E1\', \'E2\', \'E3\', \'E4\', \'E5\'], default=\'E3\')\n    p.add_argument(\'--repo-id\', default=REPO_ID)\n    p.add_argument(\'--revision\', default=REVISION, help=\'Use the same resolved HF commit for every experiment\')\n    source = p.add_mutually_exclusive_group()\n    source.add_argument(\'--local-data\', type=Path, default=DEFAULT_LOCAL_DATA,\n                        help=\'Local v5 root or data.yaml (default: dataset_crack/final_dataset_v5, relative to this script)\')\n    source.add_argument(\'--download-data\', action=\'store_true\',\n                        help=\'Explicitly use the pinned Hugging Face dataset instead of local v5\')\n    p.add_argument(\'--workspace\', type=Path, default=ROOT / \'workspace\')\n    p.add_argument(\'--project\', type=Path, default=ROOT / \'runs\')\n    p.add_argument(\'--epochs\', type=int, default=80)\n    p.add_argument(\'--imgsz\', type=int, default=640)\n    p.add_argument(\'--batch\', type=int, default=16)\n    p.add_argument(\'--workers\', type=int, default=2)\n    p.add_argument(\'--device\', default=\'0\')\n    p.add_argument(\'--seed\', type=int, default=0)\n    p.add_argument(\'--pretrained\', default=\'auto\', help=\'auto or path to matching COCO yolo26s.pt/yolo26m.pt\')\n    p.add_argument(\'--scratch\', action=\'store_true\', help=\'Explicit random initialization; not a comparable pretrained run\')\n    p.add_argument(\'--dry-run\', action=\'store_true\', help=\'Build/forward-check without downloading dataset or weights\')\n    p.add_argument(\'--allow-count-mismatch\', action=\'store_true\')\n    p.add_argument(\'--amp\', action=argparse.BooleanOptionalAction, default=True)\n    p.add_argument(\'--eval-split\', choices=[\'val\', \'test\'], default=\'val\')\n    p.add_argument(\'--evaluate-only\', type=Path, help=\'Evaluate an existing best.pt; no training\')\n    p.add_argument(\'--resume\', type=Path, help=\'Resume an interrupted last.pt in its original run directory\')\n    args = p.parse_args(argv)\n    args.local_data = None if args.download_data else args.local_data.resolve()\n    return args\n\n\ndef check_dataset_source(args):\n    """Fail locally before building a run; never silently fall back to a download."""\n    if args.local_data is None:\n        print(f\'Dataset source: Hugging Face {args.repo_id} @ {args.revision}\', flush=True)\n        return\n    source = args.local_data\n    yaml_path = source if source.is_file() else source / \'data.yaml\'\n    if not yaml_path.is_file():\n        raise FileNotFoundError(\n            f\'Local v5 dataset YAML not found: {yaml_path}. \'\n            \'Use --local-data "path/to/final_dataset_v5" or explicitly pass --download-data. \'\n            \'No dataset download was attempted.\')\n    print(f\'Dataset source: local {source}\', flush=True)\n\n\ndef run(argv=None, default_experiment=None):\n    args = arguments(argv, default_experiment)\n    if ultralytics.__version__ != \'8.4.120\':\n        raise RuntimeError(\'Install requirements.txt: architecture/parser support was tested with ultralytics==8.4.120\')\n    if args.epochs <= 0 or args.batch <= 0 or args.imgsz < 64 or args.imgsz % 32 or args.workers < 0:\n        raise ValueError(\'Positive epochs/batch, imgsz >=64 divisible by 32, workers >=0 required\')\n    if args.resume and args.evaluate_only:\n        raise ValueError(\'Choose resume OR evaluate-only\')\n    if \',\' in args.device:\n        raise ValueError(\'These notebook runners use one GPU per experiment; use separate sessions for parallel experiments\')\n    if not args.dry_run:\n        check_dataset_source(args)\n    register_modules()\n    init_seeds(args.seed, deterministic=True)\n    suffix = f\'_{args.simam_base}\' if args.experiment == \'E6\' else \'\'\n    stamp = datetime.now(timezone.utc).strftime(\'%Y%m%d_%H%M%S\')\n    name = f\'{args.experiment}{suffix}_{EXPERIMENTS[args.experiment]}_v5_normal_seed{args.seed}_{stamp}_{uuid4().hex[:6]}\'\n    work = args.workspace.resolve() / name\n    work.mkdir(parents=True, exist_ok=False)\n    weights = None if args.scratch else args.pretrained\n    if weights not in (None, \'auto\') and Path(weights).is_file():\n        weights = str(Path(weights).resolve())\n    model_yaml = write_config(args.experiment, work, args.simam_base, weights)\n    if args.dry_run:\n        model = YOLO(str(model_yaml)).model.cpu().eval()\n        with torch.no_grad():\n            model(torch.zeros(1, 3, 128, 128))\n        info = {\'experiment\': args.experiment, \'simam_base\': args.simam_base,\n                \'parameters\': sum(p.numel() for p in model.parameters()),\n                \'strides\': model.stride.tolist(), \'config\': str(model_yaml)}\n        print(json.dumps(info, indent=2))\n        return work\n    if args.device != \'cpu\' and not torch.cuda.is_available():\n        raise RuntimeError(\'Enable a GPU runtime in Colab/Kaggle, or pass --device cpu for a smoke test\')\n    data_yaml, provenance = prepare_dataset(args.workspace.resolve() / \'dataset\', args.repo_id, args.revision, args.local_data,\n                                            not args.allow_count_mismatch)\n    common = dict(data=str(data_yaml), imgsz=args.imgsz, batch=args.batch, device=args.device,\n                  workers=args.workers, plots=True, project=str(args.project.resolve()), exist_ok=False)\n    started = time.perf_counter()\n    if args.evaluate_only:\n        model = YOLO(str(args.evaluate_only.resolve()))\n        if model.model.yaml.get(\'experiment\') != args.experiment:\n            raise ValueError(\'Evaluation experiment does not match checkpoint\')\n        if args.experiment == \'E6\' and model.model.yaml.get(\'simam_base\') != args.simam_base:\n            raise ValueError(\'Pass --simam-base matching the evaluated E6 checkpoint\')\n    elif args.resume:\n        model = YOLO(str(args.resume.resolve()))\n        if model.model.yaml.get(\'experiment\') != args.experiment:\n            raise ValueError(\'Resume experiment does not match checkpoint\')\n        if not model.ckpt or model.ckpt.get(\'epoch\', -1) < 0 or model.ckpt.get(\'optimizer\') is None:\n            raise ValueError(\'Checkpoint is completed/stripped; --resume requires an interrupted last.pt with optimizer state\')\n        if args.experiment == \'E6\' and model.model.yaml.get(\'simam_base\') != args.simam_base:\n            raise ValueError(\'E6 parent must match resumed checkpoint\')\n        model.train(trainer=ExperimentTrainer, resume=True,\n                    save_dir=str(args.resume.resolve().parent.parent), **common)\n    else:\n        model = YOLO(str(model_yaml))\n        model.train(\n            trainer=ExperimentTrainer, **common, name=name, pretrained=False,\n            epochs=args.epochs, optimizer=\'AdamW\', lr0=0.002, lrf=0.01,\n            cos_lr=True, cls_pw=0.0, patience=100, close_mosaic=10,\n            seed=args.seed, deterministic=True, amp=args.amp, save_period=10,\n            # Explicit common augmentation values, independent of COCO checkpoint recipes.\n            hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, degrees=0.0, translate=0.1,\n            scale=0.5, shear=0.0, perspective=0.0, flipud=0.0, fliplr=0.5,\n            mosaic=1.0, mixup=0.0, copy_paste=0.0,\n        )\n    elapsed = time.perf_counter() - started\n    run_dir = Path(model.trainer.save_dir) if not args.evaluate_only else work\n    best = Path(model.trainer.best) if not args.evaluate_only else args.evaluate_only.resolve()\n    metadata = {\'experiment\': args.experiment, \'simam_base\': args.simam_base if args.experiment == \'E6\' else None,\n                \'dataset\': provenance, \'checkpoint\': str(best), \'elapsed_before_evaluation_seconds\': elapsed,\n                \'ultralytics\': ultralytics.__version__, \'torch\': torch.__version__, \'python\': platform.python_version(),\n                \'gpu\': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,\n                \'cli\': {k: str(v) if isinstance(v, Path) else v for k, v in vars(args).items()}}\n    (run_dir / \'experiment_metadata.json\').write_text(json.dumps(metadata, indent=2), encoding=\'utf-8\')\n    # Bundle custom modules/config so custom checkpoints remain usable after cloud sessions end.\n    support = run_dir / \'support\'\n    support.mkdir(exist_ok=True)\n    for filename in (\'modules.py\', \'architectures.py\', \'experiment_trainer.py\', \'yolo26_base.yaml\', \'requirements.txt\'):\n        shutil.copy2(ROOT / filename, support / filename)\n    shutil.copy2(model_yaml, support / model_yaml.name)\n    metrics = model.val(**common, name=name + \'_\' + args.eval_split, split=args.eval_split)\n    summary = {\'experiment\': args.experiment, \'simam_base\': args.simam_base if args.experiment == \'E6\' else None,\n               \'seed\': args.seed, \'split\': args.eval_split,\n               \'mAP50\': float(metrics.box.map50), \'mAP50_95\': float(metrics.box.map),\n               \'precision\': float(metrics.box.mp), \'recall\': float(metrics.box.mr),\n               \'speed_ms\': metrics.speed, \'checkpoint\': str(best),\n               \'per_class\': [{ \'name\': model.names[int(cls)], \'AP50\': float(metrics.box.ap50[i]),\n                              \'AP50_95\': float(metrics.box.ap[i]), \'precision\': float(metrics.box.p[i]),\n                              \'recall\': float(metrics.box.r[i])}\n                             for i, cls in enumerate(metrics.box.ap_class_index)]}\n    (Path(metrics.save_dir) / \'summary.json\').write_text(json.dumps(summary, indent=2), encoding=\'utf-8\')\n    print(json.dumps(summary, indent=2))\n    print(f\'Checkpoints and metadata: {run_dir}\')\n    print(f\'Evaluation: {metrics.save_dir}\')\n    return run_dir\n\n\nif __name__ == \'__main__\':\n    from multiprocessing import freeze_support\n    freeze_support()\n    run()\n', 'yolo26_base.yaml': '# Based on Ultralytics YOLO26, AGPL-3.0: https://ultralytics.com/license\nnc: 5\nend2end: true\nreg_max: 1\nscale: s\nscales:\n  n: [0.50, 0.25, 1024]\n  s: [0.50, 0.50, 1024]\n  m: [0.50, 1.00, 512]\n  l: [1.00, 1.00, 512]\n  x: [1.00, 1.50, 512]\nbackbone:\n  - [-1, 1, Conv, [64, 3, 2]]\n  - [-1, 1, Conv, [128, 3, 2]]\n  - [-1, 2, C3k2, [256, false, 0.25]]\n  - [-1, 1, Conv, [256, 3, 2]]\n  - [-1, 2, C3k2, [512, false, 0.25]]\n  - [-1, 1, Conv, [512, 3, 2]]\n  - [-1, 2, C3k2, [512, true]]\n  - [-1, 1, Conv, [1024, 3, 2]]\n  - [-1, 2, C3k2, [1024, true]]\n  - [-1, 1, SPPF, [1024, 5, 3, true]]\n  - [-1, 2, C2PSA, [1024]]\nhead:\n  - [-1, 1, nn.Upsample, [null, 2, nearest]]\n  - [[-1, 6], 1, Concat, [1]]\n  - [-1, 2, C3k2, [512, true]]\n  - [-1, 1, nn.Upsample, [null, 2, nearest]]\n  - [[-1, 4], 1, Concat, [1]]\n  - [-1, 2, C3k2, [256, true]]\n  - [-1, 1, Conv, [256, 3, 2]]\n  - [[-1, 13], 1, Concat, [1]]\n  - [-1, 2, C3k2, [512, true]]\n  - [-1, 1, Conv, [512, 3, 2]]\n  - [[-1, 10], 1, Concat, [1]]\n  - [-1, 1, C3k2, [1024, true, 0.5, true]]\n  - [[16, 19, 22], 1, Detect, [nc]]\n', 'requirements.txt': "ultralytics==8.4.120\nhuggingface_hub>=1.0,<3.0\nPyYAML>=6.0\nPillow>=10.0\n# Keep Colab/Kaggle's CUDA-enabled torch and torchvision installations.\n"}
for name, content in FILES.items():
    (PACKAGE / name).write_text(content, encoding='utf-8')
if str(PACKAGE) not in sys.path:
    sys.path.insert(0, str(PACKAGE))
print('Experiment code:', PACKAGE)


## 2. Install tested dependencies (keep platform GPU PyTorch)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PACKAGE / 'requirements.txt')], check=True)
import torch, ultralytics
print('Torch:', torch.__version__, '| Ultralytics:', ultralytics.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - enable GPU runtime')


## 3. Authenticate to the dataset

In [ ]:
# Add HF_TOKEN as a Colab secret or Kaggle secret with READ access to the private dataset.
# Public repositories work without a token. Never paste tokens into saved cells.
import os
if not os.getenv('HF_TOKEN'):
    try:
        if str(BASE) == '/kaggle/working':
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret('HF_TOKEN')
        elif str(BASE) == '/content':
            from google.colab import userdata
            token = userdata.get('HF_TOKEN')
        else:
            token = None
    except Exception:
        token = None
    if token:
        os.environ['HF_TOKEN'] = token
    del token
# If secrets are unavailable and the dataset is private, uncomment:
# from getpass import getpass
# os.environ['HF_TOKEN'] = getpass('HF read token: ')


## 4. Configure experiment and persistent output

In [ ]:
EXPERIMENT = 'E5'
SIMAM_BASE = 'E3'  # E6 only: E1, E2, E3, E4 or E5; choose using validation
REPO_ID = 'AkumaDachi/roadeeye-sewer-defects-v5'
REVISION = 'd88c0ed4f0fd39253eb3866bf2feb9dcf808bae5'  # Verified full-v5 snapshot; same for every experiment
EPOCHS = 80
IMGSZ = 640
BATCH = 16  # If OOM, use 8 or 4 across comparable runs; record the change
SEED = 0
WORKERS = 2
PROJECT = BASE / 'roadeeye_runs'
WORKSPACE = BASE / 'roadeeye_workspace'
LOCAL_DATA = None  # Optional already-downloaded v5 dataset root
RESUME = None  # Path to interrupted run's weights/last.pt
MOUNT_DRIVE = False  # Colab: set True to store runs/checkpoints persistently on Drive
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = Path('/content/drive/MyDrive/RoadEye/experiments')
PROJECT.mkdir(parents=True, exist_ok=True)


## 5. Download v5, verify classes/counts, fix YAML paths

In [ ]:
from dataset import prepare_dataset
DATA_YAML, DATA_INFO = prepare_dataset(WORKSPACE / 'dataset', REPO_ID, REVISION, LOCAL_DATA)
print('Class names: crack, corrosion_rust, sediment_deposit, root_intrusion, joint_defect')
print('Resolved revision:', DATA_INFO['resolved_commit'])
# Pin the resolved commit for this session; copy it into the other notebooks too.
if DATA_INFO['resolved_commit']:
    REVISION = DATA_INFO['resolved_commit']


## 6. Train and evaluate validation

In [ ]:
from run_experiment import run
RUN_ARGS = [
    '--experiment', EXPERIMENT, '--simam-base', SIMAM_BASE,
    '--repo-id', REPO_ID, '--revision', REVISION,
    '--workspace', str(WORKSPACE), '--project', str(PROJECT),
    '--epochs', str(EPOCHS), '--imgsz', str(IMGSZ), '--batch', str(BATCH),
    '--workers', str(WORKERS), '--seed', str(SEED), '--device', '0',
]
if LOCAL_DATA:
    RUN_ARGS += ['--local-data', str(LOCAL_DATA)]
else:
    RUN_ARGS += ['--download-data']  # Cloud notebooks explicitly opt into the Hub snapshot.
if RESUME:
    RUN_ARGS += ['--resume', str(RESUME)]
RUN_DIR = run(RUN_ARGS)  # Trains, then evaluates VALIDATION. No automatic test-set selection.
BEST = RUN_DIR / 'weights/best.pt'
print('Best checkpoint:', BEST)


## 7. Optional final evaluation on the full test split

In [ ]:
# Optional final full-test evaluation, after choosing configurations on validation.
RUN_FINAL_TEST = False
if RUN_FINAL_TEST:
    from modules import register_modules
    from ultralytics import YOLO
    import json
    register_modules()
    model = YOLO(str(BEST))
    metrics = model.val(data=str(DATA_YAML), split='test', imgsz=IMGSZ, batch=BATCH,
                        device=0, workers=WORKERS, plots=True, project=str(PROJECT),
                        name=RUN_DIR.name + '_full_test', exist_ok=False)
    result = {'experiment': EXPERIMENT, 'simam_base': SIMAM_BASE if EXPERIMENT == 'E6' else None,
              'seed': SEED, 'split': 'test', 'checkpoint': str(BEST), 'mAP50': float(metrics.box.map50),
              'mAP50_95': float(metrics.box.map), 'precision': float(metrics.box.mp),
              'recall': float(metrics.box.mr)}
    (Path(metrics.save_dir) / 'summary.json').write_text(json.dumps(result, indent=2))
    print(result)


## 8. Export results before ending the cloud session

In [ ]:
# Export weights, configs, CSVs, plots and custom modules. Dataset cache is excluded.
import shutil
archive_path = shutil.make_archive(str(BASE / ('results_' + RUN_DIR.name)), 'zip', str(PROJECT))
print('Save/download this before ending the runtime:', archive_path)
# Colab browser download (optional):
# from google.colab import files
# files.download(archive_path)
# Kaggle: Save Version and retain output files; download the ZIP from Output.
